In [1]:
import os
import numpy as np
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
BASE_DIR = r"c:\Users\User\Documents\GitHub\LankaLandslide-AI"

TERRAIN_RAW_DIR = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "terrain"
)

TERRAIN_PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

os.makedirs(TERRAIN_RAW_DIR, exist_ok=True)
os.makedirs(TERRAIN_PROCESSED_DIR, exist_ok=True)

print("Terrain raw folder:")
print(TERRAIN_RAW_DIR)

Terrain raw folder:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\raw\terrain


In [3]:
districts = [
    "Colombo", "Gampaha", "Kalutara",
    "Kandy", "Matale", "Nuwara Eliya",
    "Kegalle", "Ratnapura",
    "Galle", "Matara", "Hambantota",
    "Badulla", "Monaragala",
    "Kurunegala", "Puttalam",
    "Anuradhapura", "Polonnaruwa",
    "Jaffna", "Kilinochchi", "Mannar",
    "Vavuniya", "Trincomalee",
    "Batticaloa", "Ampara"
]

print("Number of districts:", len(districts))

Number of districts: 24


In [4]:
MASTER_FILE = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "landslide_weather_master.csv"
)

master = pd.read_csv(MASTER_FILE)

district_locations = (
    master[
        ["district", "latitude", "longitude"]
    ]
    .drop_duplicates("district")
    .sort_values("district")
)

print(district_locations.to_string(index=False))

    district  latitude  longitude
      Ampara    7.2917    81.6720
Anuradhapura    8.3114    80.4037
     Badulla    6.9934    81.0550
  Batticaloa    7.7170    81.7000
     Colombo    6.9271    79.8612
       Galle    6.0329    80.2168
     Gampaha    7.0840    80.0098
  Hambantota    6.1429    81.1212
      Jaffna    9.6615    80.0255
    Kalutara    6.5854    79.9607
       Kandy    7.2906    80.6337
     Kegalle    7.2513    80.3464
 Kilinochchi    9.3803    80.3770
  Kurunegala    7.4863    80.3647
      Mannar    8.9810    79.9044
      Matale    7.4675    80.6234
      Matara    5.9549    80.5550
  Monaragala    6.8728    81.3507
Nuwara Eliya    6.9497    80.7891
 Polonnaruwa    7.9403    81.0188
    Puttalam    8.0362    79.8283
   Ratnapura    6.6828    80.3992
 Trincomalee    8.5874    81.2152
    Vavuniya    8.7514    80.4971


In [5]:
!pip install rasterio requests -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import requests
import rasterio
from rasterio.io import MemoryFile
from io import BytesIO

print("Terrain libraries ready.")

Terrain libraries ready.


In [7]:
OPENTOPO_API_KEY = "91d5f2bd7212f18721619e2462383a5c"

print("API key configured.")

API key configured.


Download DEM around each district

In [8]:
import requests
import time
import os

DEM_DIR = os.path.join(
    TERRAIN_RAW_DIR,
    "dem"
)

os.makedirs(DEM_DIR, exist_ok=True)

API_URL = "https://portal.opentopography.org/API/globaldem"

# Area around each district center
BUFFER = 0.15

print("DEM folder:")
print(DEM_DIR)

DEM folder:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\raw\terrain\dem


In [9]:
test_row = district_locations[
    district_locations["district"] == "Kandy"
].iloc[0]

lat = test_row["latitude"]
lon = test_row["longitude"]

params = {
    "demtype": "COP30",
    "south": lat - BUFFER,
    "north": lat + BUFFER,
    "west": lon - BUFFER,
    "east": lon + BUFFER,
    "outputFormat": "GTiff",
    "API_Key": OPENTOPO_API_KEY
}

response = requests.get(
    API_URL,
    params=params,
    timeout=120
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("File size:", len(response.content), "bytes")

Status code: 200
Content type: application/octet-stream
File size: 4891403 bytes


In [10]:
kandy_dem = os.path.join(
    DEM_DIR,
    "Kandy_COP30.tif"
)

with open(kandy_dem, "wb") as f:
    f.write(response.content)

print("Saved:")
print(kandy_dem)

Saved:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\raw\terrain\dem\Kandy_COP30.tif


In [11]:
from pathlib import Path

kandy_dem = Path(r"C:\Users\User\Documents\GitHub\LankaLandslide-AI\data\raw\terrain\dem\Kandy_COP30.tif")

print("File exists:", kandy_dem.exists())
print("File size:", kandy_dem.stat().st_size, "bytes")

File exists: True
File size: 4891403 bytes


In [12]:
with open(kandy_dem, "rb") as f:
    header = f.read(20)

print(header)

b'II*\x00\xc0\x00\x00\x00GDAL_STRUCTU'


In [13]:
print(kandy_dem.suffix)
print(kandy_dem)

.tif
C:\Users\User\Documents\GitHub\LankaLandslide-AI\data\raw\terrain\dem\Kandy_COP30.tif


In [14]:
with open(kandy_dem, "rb") as f:
    content = f.read()

print(content.decode("utf-8", errors="ignore"))

Read the DEM

In [ ]:
import rasterio

with rasterio.open(kandy_dem) as src:
    elevation = src.read(1)
    
    print("DEM shape:", elevation.shape)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Minimum elevation:", np.nanmin(elevation))
    print("Maximum elevation:", np.nanmax(elevation))

DEM shape: (1080, 1080)
CRS: EPSG:4326
Resolution: (0.0002777777777777778, 0.0002777777777777778)
Minimum elevation: 148.39136
Maximum elevation: 1761.7133


Calculate slope

In [ ]:
from rasterio.transform import xy

with rasterio.open(kandy_dem) as src:
    elevation = src.read(1).astype(float)
    transform = src.transform
    resolution_x = abs(transform.a)
    resolution_y = abs(transform.e)

# Approximate geographic resolution in degrees
# Convert degrees to meters around Sri Lanka
lat_reference = 7.3

meters_per_degree_lat = 111320
meters_per_degree_lon = (
    111320 * np.cos(np.radians(lat_reference))
)

dx = resolution_x * meters_per_degree_lon
dy = resolution_y * meters_per_degree_lat

dz_dy, dz_dx = np.gradient(
    elevation,
    dy,
    dx
)

slope_radians = np.arctan(
    np.sqrt(
        dz_dx**2 +
        dz_dy**2
    )
)

slope_degrees = np.degrees(
    slope_radians
)

print("Mean elevation:", round(np.nanmean(elevation), 2), "m")
print("Mean slope:", round(np.nanmean(slope_degrees), 2), "degrees")
print("Maximum slope:", round(np.nanmax(slope_degrees), 2), "degrees")

Mean elevation: 614.27 m
Mean slope: 16.0 degrees
Maximum slope: 69.19 degrees


Create Kandy terrain record

In [ ]:
kandy_terrain = pd.DataFrame([{
    "district": "Kandy",
    "elevation_mean_m": np.nanmean(elevation),
    "elevation_max_m": np.nanmax(elevation),
    "elevation_min_m": np.nanmin(elevation),
    "slope_mean_deg": np.nanmean(slope_degrees),
    "slope_max_deg": np.nanmax(slope_degrees)
}])

kandy_terrain

,district,elevation_mean_m,elevation_max_m,elevation_min_m,slope_mean_deg,slope_max_deg
0,Kandy,614.273682,1761.713257,148.391357,15.997349,69.188167


Download and calculate terrain for all distric

In [ ]:
terrain_records = []

for _, row in district_locations.iterrows():

    district = row["district"]
    lat = row["latitude"]
    lon = row["longitude"]

    print(f"Processing: {district}")

    dem_file = os.path.join(
        DEM_DIR,
        f"{district}_COP30.tif"
    )

    try:
        # Download DEM
        params = {
            "demtype": "COP30",
            "south": lat - BUFFER,
            "north": lat + BUFFER,
            "west": lon - BUFFER,
            "east": lon + BUFFER,
            "outputFormat": "GTiff",
            "API_Key": OPENTOPO_API_KEY
        }

        response = requests.get(
            API_URL,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        with open(dem_file, "wb") as f:
            f.write(response.content)

        # Read DEM
        with rasterio.open(dem_file) as src:

            elevation = src.read(1).astype(float)

            transform = src.transform

            resolution_x = abs(transform.a)
            resolution_y = abs(transform.e)

        # Convert approximate degree resolution to metres
        meters_per_degree_lat = 111320

        meters_per_degree_lon = (
            111320 * np.cos(np.radians(lat))
        )

        dx = resolution_x * meters_per_degree_lon
        dy = resolution_y * meters_per_degree_lat

        # Calculate slope
        dz_dy, dz_dx = np.gradient(
            elevation,
            dy,
            dx
        )

        slope_radians = np.arctan(
            np.sqrt(
                dz_dx**2 +
                dz_dy**2
            )
        )

        slope_degrees = np.degrees(
            slope_radians
        )

        # Store terrain statistics
        terrain_records.append({
            "district": district,

            "elevation_mean_m":
                np.nanmean(elevation),

            "elevation_max_m":
                np.nanmax(elevation),

            "elevation_min_m":
                np.nanmin(elevation),

            "slope_mean_deg":
                np.nanmean(slope_degrees),

            "slope_max_deg":
                np.nanmax(slope_degrees)
        })

        print("  ✓ Success")

        time.sleep(1)

    except Exception as e:

        print(
            f"  ✗ Failed: {district}"
        )
        print("  Error:", e)

print("\nTerrain collection completed.")

Processing: Ampara
  ✓ Success
Processing: Anuradhapura
  ✓ Success
Processing: Badulla
  ✓ Success
Processing: Batticaloa
  ✓ Success
Processing: Colombo
  ✓ Success
Processing: Galle
  ✓ Success
Processing: Gampaha
  ✓ Success
Processing: Hambantota
  ✓ Success
Processing: Jaffna
  ✓ Success
Processing: Kalutara
  ✓ Success
Processing: Kandy
  ✓ Success
Processing: Kegalle
  ✓ Success
Processing: Kilinochchi
  ✓ Success
Processing: Kurunegala
  ✓ Success
Processing: Mannar
  ✓ Success
Processing: Matale
  ✓ Success
Processing: Matara
  ✓ Success
Processing: Monaragala
  ✓ Success
Processing: Nuwara Eliya
  ✓ Success
Processing: Polonnaruwa
  ✓ Success
Processing: Puttalam
  ✓ Success
Processing: Ratnapura
  ✓ Success
Processing: Trincomalee
  ✓ Success
Processing: Vavuniya
  ✓ Success

Terrain collection completed.


Create terrain dataframe

In [ ]:
terrain_df = pd.DataFrame(
    terrain_records
)

print("Terrain dataset shape:")
print(terrain_df.shape)

display(terrain_df)

Terrain dataset shape:
(24, 6)


,district,elevation_mean_m,elevation_max_m,elevation_min_m,slope_mean_deg,slope_max_deg
0,Ampara,45.176092,615.203064,-3.557961,3.358947,66.363210
1,Anuradhapura,96.962870,393.882843,58.273411,2.220605,50.197250
2,Badulla,871.198839,2034.031616,143.385620,19.920840,68.916145
3,Batticaloa,4.934191,101.875320,-3.020253,0.667344,22.039244
4,Colombo,7.889106,137.542725,-9.874548,2.067448,49.814870
5,Galle,14.947572,283.317108,-11.264187,3.962113,47.622135
6,Gampaha,28.100720,222.210526,-6.218959,5.170570,43.916338
7,Hambantota,14.318886,114.409737,-3.729290,1.130885,29.348897
8,Jaffna,2.646899,24.972681,-4.791825,0.828800,17.871005
9,Kalutara,12.454278,282.763031,-1.644879,3.837043,56.865051


In [ ]:
print("Missing values:")
print(terrain_df.isnull().sum())

print("\nNumber of districts:")
print(terrain_df["district"].nunique())

Missing values:
district            0
elevation_mean_m    0
elevation_max_m     0
elevation_min_m     0
slope_mean_deg      0
slope_max_deg       0
dtype: int64

Number of districts:
24


Save the terrain dataset

In [ ]:
TERRAIN_OUTPUT = os.path.join(
    TERRAIN_PROCESSED_DIR,
    "district_terrain_features.csv"
)

terrain_df.to_csv(
    TERRAIN_OUTPUT,
    index=False
)

print("Terrain dataset saved:")
print(TERRAIN_OUTPUT)

Terrain dataset saved:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\processed\district_terrain_features.csv


Final terrain check

In [ ]:
print(
    terrain_df.describe()
)

       elevation_mean_m  elevation_max_m  elevation_min_m  slope_mean_deg  \
count         24.000000        24.000000        24.000000       24.000000   
mean         203.844120       719.328560        44.924795        6.496278   
std          356.898269       790.342749        98.401887        6.455796   
min            1.903507        24.972681       -11.264187        0.377996   
25%           13.852734       131.759478        -3.994924        1.584624   
50%           36.638406       288.588974        -1.874568        3.597995   
75%          222.334329      1149.237549        52.528906       10.812150   
max         1498.647701      2523.122070       445.743164       19.920840   

       slope_max_deg  
count      24.000000  
mean       48.988478  
std        19.704978  
min        16.340579  
25%        33.799461  
50%        50.006060  
75%        66.695917  
max        72.377657  


Terrain quality check

In [ ]:
print("Terrain quality checks")
print("=" * 50)

print("Number of districts:", terrain_df["district"].nunique())

print("\nMissing values:")
print(terrain_df.isnull().sum())

print("\nDuplicate districts:")
print(terrain_df["district"].duplicated().sum())

print("\nNegative elevation values:")
print(
    terrain_df[
        terrain_df["elevation_min_m"] < 0
    ][
        ["district", "elevation_min_m"]
    ]
)

print("\nInvalid slope values:")
print(
    terrain_df[
        (terrain_df["slope_mean_deg"] < 0) |
        (terrain_df["slope_max_deg"] < 0)
    ]
)

print("\nTerrain statistics:")
print(
    terrain_df[
        [
            "elevation_mean_m",
            "elevation_max_m",
            "elevation_min_m",
            "slope_mean_deg",
            "slope_max_deg"
        ]
    ].describe()
)

Terrain quality checks
Number of districts: 24

Missing values:
district            0
elevation_mean_m    0
elevation_max_m     0
elevation_min_m     0
slope_mean_deg      0
slope_max_deg       0
dtype: int64

Duplicate districts:
0

Negative elevation values:
       district  elevation_min_m
0        Ampara        -3.557961
3    Batticaloa        -3.020253
4       Colombo        -9.874548
5         Galle       -11.264187
6       Gampaha        -6.218959
7    Hambantota        -3.729290
8        Jaffna        -4.791825
9      Kalutara        -1.644879
12  Kilinochchi        -3.115343
14       Mannar        -2.104258
16       Matara        -2.708204
20     Puttalam        -6.610041
22  Trincomalee        -5.528057

Invalid slope values:
Empty DataFrame
Columns: [district, elevation_mean_m, elevation_max_m, elevation_min_m, slope_mean_deg, slope_max_deg]
Index: []

Terrain statistics:
       elevation_mean_m  elevation_max_m  elevation_min_m  slope_mean_deg  \
count         24.000000    

Check for impossible terrain relationships

In [ ]:
invalid_relationships = terrain_df[
    (terrain_df["elevation_min_m"] >
     terrain_df["elevation_mean_m"]) |
    (terrain_df["elevation_mean_m"] >
     terrain_df["elevation_max_m"]) |
    (terrain_df["slope_mean_deg"] >
     terrain_df["slope_max_deg"])
]

print("Invalid relationships found:")
print(len(invalid_relationships))

if len(invalid_relationships) > 0:
    display(invalid_relationships)
else:
    print("✓ All terrain relationships are valid.")

Invalid relationships found:
0
✓ All terrain relationships are valid.


In [ ]:
TERRAIN_OUTPUT = os.path.join(
    TERRAIN_PROCESSED_DIR,
    "district_terrain_features.csv"
)

terrain_df.to_csv(
    TERRAIN_OUTPUT,
    index=False
)

print("Terrain dataset saved:")
print(TERRAIN_OUTPUT)

Terrain dataset saved:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\processed\district_terrain_features.csv


Load the engineered dataset

In [ ]:
FEATURE_FILE = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "landslide_weather_features.csv"
)

features_df = pd.read_csv(FEATURE_FILE)

features_df["date"] = pd.to_datetime(
    features_df["date"]
)

print("Weather feature dataset:")
print(features_df.shape)

Weather feature dataset:
(192120, 32)


Merge terrain features

In [ ]:
combined_df = features_df.merge(
    terrain_df,
    on="district",
    how="left"
)

print("Combined dataset shape:")
print(combined_df.shape)

print("\nMissing values after merge:")
print(combined_df.isnull().sum())

Combined dataset shape:
(192120, 37)

Missing values after merge:
date                  0
rainfall_mm           0
rain_mm               0
rain_hours            0
temp_max              0
temp_min              0
temp_mean             0
wind_max              0
wind_gust_max         0
district              0
latitude              0
longitude             0
landslide_events      0
landslide_recorded    0
rainfall_1day         0
rainfall_3day         0
rainfall_7day         0
rainfall_14day        0
rainfall_30day        0
rainfall_3day_avg     0
rainfall_7day_avg     0
rainfall_14day_avg    0
rainfall_30day_avg    0
temperature_range     0
month                 0
day_of_year           0
month_sin             0
month_cos             0
heavy_rain_10mm       0
heavy_rain_25mm       0
heavy_rain_50mm       0
heavy_rain_100mm      0
elevation_mean_m      0
elevation_max_m       0
elevation_min_m       0
slope_mean_deg        0
slope_max_deg         0
dtype: int64


Verify all 24 districts

In [ ]:
print("Districts:", combined_df["district"].nunique())

print("\nDistricts missing terrain:")
print(
    combined_df[
        combined_df["elevation_mean_m"].isna()
    ]["district"].unique()
)

Districts: 24

Districts missing terrain:
[]


Check the new features

In [ ]:
terrain_features = [
    "elevation_mean_m",
    "elevation_max_m",
    "elevation_min_m",
    "slope_mean_deg",
    "slope_max_deg"
]

print(
    combined_df[
        ["district"] + terrain_features
    ].drop_duplicates()
    .sort_values("district")
    .to_string(index=False)
)

    district  elevation_mean_m  elevation_max_m  elevation_min_m  slope_mean_deg  slope_max_deg
      Ampara         45.176092       615.203064        -3.557961        3.358947      66.363210
Anuradhapura         96.962870       393.882843        58.273411        2.220605      50.197250
     Badulla        871.198839      2034.031616       143.385620       19.920840      68.916145
  Batticaloa          4.934191       101.875320        -3.020253        0.667344      22.039244
     Colombo          7.889106       137.542725        -9.874548        2.067448      49.814870
       Galle         14.947572       283.317108       -11.264187        3.962113      47.622135
     Gampaha         28.100720       222.210526        -6.218959        5.170570      43.916338
  Hambantota         14.318886       114.409737        -3.729290        1.130885      29.348897
      Jaffna          2.646899        24.972681        -4.791825        0.828800      17.871005
    Kalutara         12.454278       282

Save the combined dataset

In [ ]:
COMBINED_OUTPUT = os.path.join(
    BASE_DIR,
    "data",
    "features",
    "landslide_weather_terrain_features.csv"
)

os.makedirs(
    os.path.dirname(COMBINED_OUTPUT),
    exist_ok=True
)

combined_df.to_csv(
    COMBINED_OUTPUT,
    index=False
)

print("Combined dataset saved:")
print(COMBINED_OUTPUT)

print("\nFinal shape:", combined_df.shape)

Combined dataset saved:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\features\landslide_weather_terrain_features.csv

Final shape: (192120, 37)
